# IMDB Sentiment Classification: RNN vs LSTM

A deep learning project that compares a **Simple Recurrent Neural Network (RNN)** and a **Long Short-Term Memory (LSTM)** model for binary sentiment classification on the IMDB movie review dataset.

## Objective
Study the working of LSTM networks and compare them with a traditional RNN for sentiment classification.

## Tasks Covered
- Load and preprocess IMDB movie reviews.
- Clean text using lowercasing, HTML/URL removal, punctuation and number removal, stopword removal, emoji removal, and stemming.
- Convert reviews into numerical TF-IDF features.
- Split the data into training, validation, and test sets.
- Build and train a Simple RNN.
- Build and train an LSTM.
- Evaluate both models using accuracy, precision, recall, F1-score, classification reports, confusion matrices, and training time.
- Compare the recorded RNN and LSTM metrics.

## Models
Both models use a two-layer recurrent architecture with 128 hidden units, dropout of 0.3, followed by fully connected layers for binary sentiment classification.

## Technologies
Python, PyTorch, Pandas, NumPy, NLTK, Scikit-learn, and TensorFlow/Keras preprocessing utilities.

> **Implementation note:** The uploaded notebook represents reviews as 5,000-dimensional TF-IDF vectors and feeds each vector as a single sequence step (`unsqueeze(1)`) into the RNN/LSTM. This README describes the implementation as it exists in the notebook rather than replacing it with a different token-sequence architecture.


# IMDB Sentiment Classification: RNN vs LSTM



In [ ]:
import numpy as np
import pandas as pd
import re
import string
from tensorflow.keras.preprocessing.sequence import pad_sequences


## 1. Load the IMDB Dataset


In [ ]:
df=pd.read_csv("/content/IMDB Dataset.csv", engine='python', on_bad_lines='skip')

## 2. Inspect Dataset Shape


In [ ]:
df.shape

(50000, 2)

## 3. Preview the Reviews


In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 4. Inspect Dataset Information


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


## 5. Check for Missing Values


In [ ]:
df.isnull().sum()


,0
review,0
sentiment,0


## 6. Remove Duplicate Reviews


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(49582, 2)

## 7. Encode Sentiment Labels


In [ ]:
df['sentiment'] = df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


## 8. Convert Reviews to Lowercase


In [ ]:
df['review'] = df['review'].str.lower()

## 9. Remove HTML Tags


In [ ]:
def remove_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

df['review'] = df['review'].apply(remove_html)

## 10. Remove URLs


In [ ]:
def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

df['review'] = df['review'].apply(remove_url)

## 11. Remove Punctuation


In [ ]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['review'] = df['review'].apply(remove_punctuation)

## 12. Remove Numbers


In [ ]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

df['review'] = df['review'].apply(remove_numbers)

## 13. Normalize Whitespace


In [ ]:
def remove_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

df['review'] = df['review'].apply(remove_spaces)

## 14. Load Stopwords


In [ ]:
from nltk.corpus import stopwords


In [ ]:
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 15. Remove Stopwords


In [ ]:
def remove_stopwords(text):
    new_text = []

    for word in text.split():
        if word not in stop_words:
            new_text.append(word)

    return " ".join(new_text)

In [ ]:
print(type(df))
df['review']=df['review'].apply(remove_stopwords)
print(df.head())

<class 'pandas.core.frame.DataFrame'>
                                              review  sentiment
0  one reviewers mentioned watching oz episode yo...          1
1  wonderful little production filming technique ...          1
2  thought wonderful way spend time hot summer we...          1
3  basically theres family little boy jake thinks...          0
4  petter matteis love time money visually stunni...          1


## 16. Remove Emojis


In [ ]:
import re
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

In [ ]:
!pip install emoji
import emoji
print(emoji.demojize('Python is 🔥'))

Python is :fire:


In [ ]:


from nltk.stem import PorterStemmer
df.head()

,review,sentiment
0,one review mention watch oz episod youll hook ...,1
1,wonder littl product film techniqu unassum old...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


## 17. Apply Stemming


In [ ]:
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

ps = PorterStemmer()

def stemming(text):
    tokens = word_tokenize(text)
    stemmed_words = [ps.stem(token) for token in tokens]
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

## 18. Create TF-IDF Features


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4035808 stored elements and shape (49582, 5000)>

In [ ]:
X = df["review"]
y = df["sentiment"]

## 19. Train-Test Split


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 20. Training-Validation Split


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
X_train.shape

(31732,)

## 21. Convert Features and Labels to PyTorch Tensors


In [ ]:
import torch

# Transform the text data into numerical TF-IDF features
X_train_tf = tf.transform(X_train).toarray()
X_val_tf = tf.transform(X_val).toarray()
X_test_tf = tf.transform(X_test).toarray()

# Convert numerical features to PyTorch tensors with float32 dtype
X_train_tensor = torch.tensor(X_train_tf, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_tf, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_tf, dtype=torch.float32)

# Convert labels to PyTorch tensors with float32 dtype (common for binary classification targets)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

## 22. Create Tensor Datasets


In [ ]:
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

## 23. Create DataLoaders


In [ ]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)

val_loader = DataLoader( val_dataset, batch_size=batch_size,shuffle=False)

test_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

## 24. Select Computation Device


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## 25. Build the Simple RNN Model


In [ ]:
import torch
import torch.nn as nn

class SentimentRNN(nn.Module):

    def __init__(self, input_size):
        super(SentimentRNN, self).__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.fc1 = nn.Linear(128, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        out, hidden = self.rnn(x)

        out = out[:, -1, :]      # Last time step

        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        return out

## 26. Initialize the RNN Model


In [ ]:
input_size = X_train_tensor.shape[1]

model = SentimentRNN(input_size=input_size).to(device)

print(model)

SentimentRNN(
  (rnn): RNN(5000, 128, num_layers=2, batch_first=True, dropout=0.3)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


## 27. Define RNN Loss Function and Optimizer


In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## 28. Train and Validate the RNN


In [ ]:
import torch
import time

rnn_start_time = time.time()

epochs = 10

rnn_train_losses = []
rnn_val_losses = []

rnn_train_accuracies = []
rnn_val_accuracies = []

for epoch in range(epochs):

    #TRAIN
    model.train()

    running_train_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.unsqueeze(1).to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        outputs = outputs.squeeze(1)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()

        total += y_batch.size(0)

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    rnn_train_losses.append(train_loss)
    rnn_train_accuracies.append(train_accuracy)

    #VALIDATION

    model.eval()

    running_val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.unsqueeze(1).to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            outputs = outputs.squeeze(1)

            loss = criterion(outputs, y_batch)

            running_val_loss += loss.item()

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            correct += (predictions == y_batch).sum().item()

            total += y_batch.size(0)

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    rnn_val_losses.append(val_loss)
    rnn_val_accuracies.append(val_accuracy)

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy:.2f}%")
    print(f"Val Loss : {val_loss:.4f}")
    print(f"Val Accuracy : {val_accuracy:.2f}%")
    print("-"*60)

rnn_end_time = time.time()

rnn_training_time = rnn_end_time-rnn_start_time

print(f"RNN Training Time : {rnn_training_time:.2f} seconds")

Epoch [1/10]
Train Loss : 0.0634
Train Accuracy : 97.44%
Val Loss : 0.8342
Val Accuracy : 85.15%
------------------------------------------------------------
Epoch [2/10]
Train Loss : 0.0596
Train Accuracy : 97.61%
Val Loss : 0.8582
Val Accuracy : 84.61%
------------------------------------------------------------
Epoch [3/10]
Train Loss : 0.0490
Train Accuracy : 98.04%
Val Loss : 1.0340
Val Accuracy : 84.32%
------------------------------------------------------------
Epoch [4/10]
Train Loss : 0.0471
Train Accuracy : 98.19%
Val Loss : 1.0872
Val Accuracy : 84.51%
------------------------------------------------------------
Epoch [5/10]
Train Loss : 0.0409
Train Accuracy : 98.54%
Val Loss : 1.0400
Val Accuracy : 84.60%
------------------------------------------------------------
Epoch [6/10]
Train Loss : 0.0381
Train Accuracy : 98.68%
Val Loss : 1.0289
Val Accuracy : 84.28%
------------------------------------------------------------
Epoch [7/10]
Train Loss : 0.0339
Train Accuracy : 98

## 29. Evaluate RNN Test Accuracy


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.unsqueeze(1).to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        predictions = (torch.sigmoid(outputs.squeeze()) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

rnn_test_accuracy = (correct / total) * 100

print(f"Rnn Test Accuracy : {rnn_test_accuracy:.2f}%")

Rnn Test Accuracy : 84.42%


## 30. Evaluate RNN: Precision, Recall, F1 and Confusion Matrix


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.unsqueeze(1).to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        predictions = (torch.sigmoid(outputs.squeeze()) >= 0.5).float()

        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

rnn_accuracy = accuracy_score(y_true, y_pred)
rnn_precision = precision_score(y_true, y_pred)
rnn_recall = recall_score(y_true, y_pred)
rnn_f1 = f1_score(y_true, y_pred)

print(f"Rnn Accuracy  : {rnn_accuracy:.4f}")
print(f"Rnn Precision : {rnn_precision:.4f}")
print(f"Rnn Recall    : {rnn_recall:.4f}")
print(f"Rnn F1 Score  : {rnn_f1:.4f}")

print("\nRnn_Classification Report\n")
print(classification_report(y_true, y_pred))

rnn_cm = confusion_matrix(y_true, y_pred)
print("\nRnn_Confusion Matrix\n")
print(rnn_cm)

Rnn Accuracy  : 0.8485
Rnn Precision : 0.8731
Rnn Recall    : 0.8170
Rnn F1 Score  : 0.8441

Rnn_Classification Report

              precision    recall  f1-score   support

         0.0       0.83      0.88      0.85      4940
         1.0       0.87      0.82      0.84      4977

    accuracy                           0.85      9917
   macro avg       0.85      0.85      0.85      9917
weighted avg       0.85      0.85      0.85      9917


Rnn_Confusion Matrix

[[4349  591]
 [ 911 4066]]


## 31. Build the LSTM Model


LSTM MODEL


In [ ]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):

    def __init__(self, input_size):
        super(SentimentLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.fc1 = nn.Linear(128, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        out = output[:, -1, :]

        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.fc2(out)

        return out

## 32. Initialize the LSTM Model


In [ ]:
input_size = X_train_tensor.shape[1]

model = SentimentLSTM(input_size).to(device)

print(model)

SentimentLSTM(
  (lstm): LSTM(5000, 128, num_layers=2, batch_first=True, dropout=0.3)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


## 33. Define LSTM Loss Function


In [ ]:
criterion = nn.BCEWithLogitsLoss()

## 34. Define LSTM Optimizer


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## 35. Train and Validate the LSTM


In [ ]:
import torch
import time

lstm_start_time = time.time()

epochs = 10

lstm_train_losses = []
lstm_val_losses = []

lstm_train_accuracies = []
lstm_val_accuracies = []

for epoch in range(epochs):

    model.train()

    running_train_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.unsqueeze(1).to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        outputs = outputs.squeeze(1)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        predictions = (torch.sigmoid(outputs)>=0.5).float()

        correct += (predictions==y_batch).sum().item()

        total += y_batch.size(0)

    train_loss = running_train_loss/len(train_loader)
    train_accuracy = 100*correct/total

    lstm_train_losses.append(train_loss)
    lstm_train_accuracies.append(train_accuracy)

    #VALIDATION

    model.eval()

    running_val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch,y_batch in val_loader:

            X_batch=X_batch.unsqueeze(1).to(device)
            y_batch=y_batch.to(device)

            outputs=model(X_batch)

            outputs=outputs.squeeze(1)

            loss=criterion(outputs,y_batch)

            running_val_loss += loss.item()

            predictions=(torch.sigmoid(outputs)>=0.5).float()

            correct += (predictions==y_batch).sum().item()

            total += y_batch.size(0)

    val_loss = running_val_loss/len(val_loader)
    val_accuracy = 100*correct/total

    lstm_val_losses.append(val_loss)
    lstm_val_accuracies.append(val_accuracy)

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy:.2f}%")
    print(f"Val Loss : {val_loss:.4f}")
    print(f"Val Accuracy : {val_accuracy:.2f}%")
    print("-"*60)

lstm_end_time = time.time()

lstm_training_time = lstm_end_time-lstm_start_time

print(f"LSTM Training Time : {lstm_training_time:.2f} seconds")

Epoch [1/10]
Train Loss : 0.3435
Train Accuracy : 84.65%
Val Loss : 0.2831
Val Accuracy : 87.97%
------------------------------------------------------------
Epoch [2/10]
Train Loss : 0.2418
Train Accuracy : 90.47%
Val Loss : 0.2915
Val Accuracy : 87.96%
------------------------------------------------------------
Epoch [3/10]
Train Loss : 0.2157
Train Accuracy : 91.42%
Val Loss : 0.3033
Val Accuracy : 87.60%
------------------------------------------------------------
Epoch [4/10]
Train Loss : 0.1865
Train Accuracy : 92.29%
Val Loss : 0.3526
Val Accuracy : 86.49%
------------------------------------------------------------
Epoch [5/10]
Train Loss : 0.1626
Train Accuracy : 92.81%
Val Loss : 0.3943
Val Accuracy : 86.81%
------------------------------------------------------------
Epoch [6/10]
Train Loss : 0.1386
Train Accuracy : 93.60%
Val Loss : 0.4656
Val Accuracy : 86.60%
------------------------------------------------------------
Epoch [7/10]
Train Loss : 0.1222
Train Accuracy : 94

## 36. Evaluate LSTM Test Accuracy


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.unsqueeze(1).to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        outputs = outputs.squeeze(1)

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

lstm_test_accuracy = 100 * correct / total

print(f"LSTM Test Accuracy : {lstm_test_accuracy:.2f}%")

LSTM Test Accuracy : 84.85%


## 37. Evaluate LSTM: Precision, Recall and F1


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.unsqueeze(1).to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        outputs = outputs.squeeze(1)

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

lstm_accuracy = accuracy_score(y_true, y_pred)
lstm_precision = precision_score(y_true, y_pred)
lstm_recall = recall_score(y_true, y_pred)
lstm_f1 = f1_score(y_true, y_pred)

print(f"Accuracy  : {lstm_accuracy*100:.2f}%")
print(f"Precision : {lstm_precision:.4f}")
print(f"Recall    : {lstm_recall:.4f}")
print(f"F1 Score  : {lstm_f1:.4f}")

Accuracy  : 84.85%
Precision : 0.8731
Recall    : 0.8170
F1 Score  : 0.8441


## 38. LSTM Classification Report


In [ ]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

         0.0       0.83      0.88      0.85      4940
         1.0       0.87      0.82      0.84      4977

    accuracy                           0.85      9917
   macro avg       0.85      0.85      0.85      9917
weighted avg       0.85      0.85      0.85      9917



## 39. LSTM Confusion Matrix


In [ ]:
lstm_cm = confusion_matrix(y_true, y_pred)

print(lstm_cm)

[[4349  591]
 [ 911 4066]]


## 40. Compare RNN and LSTM


In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": [
        "Training Accuracy",
        "Validation Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Training Time (sec)"
    ],

    "RNN": [
        rnn_train_accuracies[-1],
        rnn_val_accuracies[-1],
        rnn_test_accuracy,
        rnn_precision,
        rnn_recall,
        rnn_f1,
        rnn_training_time
    ],

    "LSTM": [
        lstm_train_accuracies[-1],
        lstm_val_accuracies[-1],
        lstm_test_accuracy,
        lstm_precision,
        lstm_recall,
        lstm_f1,
        lstm_training_time
    ]
})

print(comparison)

                Metric         RNN        LSTM
0    Training Accuracy   99.322451   95.915795
1  Validation Accuracy   84.154796   84.356486
2        Test Accuracy   84.420692   84.854291
3            Precision    0.873094    0.873094
4               Recall    0.816958    0.816958
5             F1 Score    0.844094    0.844094
6  Training Time (sec)  880.710457  742.887143


## Conclusion

This notebook provides an experimental comparison between Simple RNN and LSTM models for IMDB sentiment classification using the same preprocessed feature representation and training setup. The final comparison table reports the metrics captured during the notebook execution.